## 06. Model Evaluation

In [1]:
from pathlib import Path
import time
import joblib
import sys

sys.path.append(str(Path.cwd().parents[0]))
from src.validation import *
from src.tuning import show_metrics

In [2]:
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.ensemble import RandomForestClassifier
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.base import BaseEstimator
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# load the data
df = pd.read_csv('../data/processed/sms_cleaned_data.csv')
df = df[(df['target'] == 0) | (df['target'] == 1)]
df

,date,text,carrier,target,punctuation_count,text_count,has_url,has_money_mention,has_phone_in_text,exclamation_count,uppercase_ratio,word_count,year,month
2,2026-03-27 06:23:44.260,papalduhin kita moneytoken ngayon basta bahala...,Smart,1,6,167,0.0,1.0,0.0,0.0,0.071856,22,2026,March
3,2026-02-17 10:25:35.838,boss panalunin kita pinata win moneytoken haba...,Smart,1,11,186,0.0,1.0,0.0,0.0,0.016129,20,2026,February
5,2025-11-15 10:25:41.961,okay makipag deal sayo cash ipapanalo kita mon...,Smart,1,10,195,0.0,1.0,0.0,0.0,0.020513,25,2025,November
7,2025-08-27 09:28:23.142,fex parcel p iisddcddtba reach please contact ...,Smart,1,5,109,0.0,0.0,1.0,0.0,0.119266,10,2025,August
8,2025-08-19 08:57:21.957,smart reminds reward points expire today pleas...,Unknown,1,11,114,1.0,0.0,0.0,0.0,0.043860,11,2025,August
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999,2018-11-09 01:33:50.801,properties bgc makati moa san juan binondo pas...,Smart,0,9,195,0.0,0.0,0.0,1.0,0.220513,20,2018,November
1010,2018-05-06 04:37:35.505,available instantly ageless effective proven e...,Smart,0,7,148,0.0,0.0,0.0,0.0,0.162162,15,2018,May
1013,2018-04-19 11:11:20.770,co makati needs software java developer bachel...,Globe,0,8,171,1.0,0.0,0.0,0.0,0.058480,17,2018,April
1014,2018-03-22 10:45:57.147,co makati needs software java developer bachel...,Smart,0,9,187,1.0,0.0,0.0,0.0,0.053476,19,2018,March


In [3]:
df['dedup_group_id'] = pd.factorize(df['text'])[0]

binary_cols = ['has_url', 'has_money_mention', 'has_phone_in_text']
numericals = ['exclamation_count', 'text_count', 'word_count',
              'punctuation_count', 'uppercase_ratio']

x = df['text']
y = df['target']

# splitting and vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.8, sublinear_tf=True)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
scaler = MinMaxScaler(clip=True)
train_idx, test_idx = next(sgkf.split(x, y, groups=df['dedup_group_id']))

train_data = df.iloc[train_idx].reset_index(drop=True)
test_data = df.iloc[test_idx].reset_index(drop=True)

X_train = vectorizer.fit_transform(train_data['text'])
X_test = vectorizer.transform(test_data['text'])
y_train = train_data['target']
y_test = test_data['target']

num_train = scaler.fit_transform(train_data[numericals])
num_test  = scaler.transform(test_data[numericals])

bin_train = train_data[binary_cols].to_numpy(dtype=float)
bin_test  = test_data[binary_cols].to_numpy(dtype=float)

X_train = hstack([X_train, csr_matrix(bin_train), csr_matrix(num_train)]).tocsr()
X_test  = hstack([X_test,  csr_matrix(bin_test),  csr_matrix(num_test)]).tocsr()

# init both naive bayes model and using hyperparamters
mnb = MultinomialNB(alpha=0.1, fit_prior=False)
cnb = ComplementNB(alpha=0.1, norm=True)

mnb.fit(X_train, y_train)
cnb.fit(X_train, y_train)

ypred_mnb = mnb.predict(X_test)
ypred_cnb = cnb.predict(X_test)

yprob_mnb = mnb.predict_proba(X_test)
yprob_cnb = cnb.predict_proba(X_test)

rf = RandomForestClassifier(
    class_weight=None, criterion='entropy', max_depth=100,
    max_features='log2', min_samples_leaf=1, min_samples_split=4, n_estimators=200,
    random_state=42  
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)

print(f"Training Data Size: {len(train_data)} | Test Data Size: {len(test_data)}")

Training Data Size: 545 | Test Data Size: 137


In [4]:
# helper function for checking predictions
def prediction_check(model: BaseEstimator, data: pd.DataFrame) -> pd.DataFrame:

    texts = data['text'].values
    predicted = model.predict(texts)
    actuals = data['target'].values
    matched = (actuals == predicted).astype('int')

    pred_check = pd.DataFrame({
        'Text' :  texts,
        'Actual' : actuals,
        'Predicted' : predicted,
        "Is Matched" : matched
    })

    return pred_check

## Model Evaluation for CNB and MNB

In [9]:
names = build_feature_names(vectorizer, binary_cols, numericals)
stats = term_stats(X_train, y_train, names)

nb_tables = {
    'MNB': nb_word_table(mnb, feature_names=names, stats=stats, top_n=15, max_ngram=1),
    'CNB': nb_word_table(cnb, feature_names=names, stats=stats, top_n=15, max_ngram=1),
}

nb_tables['MNB'].head(10)

,feature,rank,kind,docs_Smishing,mean_Smishing,docs_Legitimate,mean_Legitimate,lean,score,class
0,redeem,1,word,0,0.0,22,0.012612,Legitimate,3.291971,Legitimate
1,gojackpot,2,word,0,0.0,24,0.012481,Legitimate,3.281786,Legitimate
2,points,3,word,0,0.0,20,0.010899,Legitimate,3.149731,Legitimate
3,reminds,4,word,0,0.0,18,0.009625,Legitimate,3.028981,Legitimate
4,epicwin,5,word,0,0.0,19,0.009414,Legitimate,3.007473,Legitimate
5,login,6,word,0,0.0,12,0.009400,Legitimate,3.006054,Legitimate
6,trusted,7,word,0,0.0,19,0.008836,Legitimate,2.946160,Legitimate
7,tangkilikin,8,word,0,0.0,15,0.007871,Legitimate,2.834555,Legitimate
8,soon,9,word,0,0.0,13,0.007684,Legitimate,2.811448,Legitimate
9,smart,10,word,0,0.0,12,0.007489,Legitimate,2.786703,Legitimate


In [10]:
nb_tables['CNB'].head(10)

,feature,rank,kind,docs_Smishing,mean_Smishing,docs_Legitimate,mean_Legitimate,lean,score,class
0,redeem,1,word,0,0.0,22,0.012612,Legitimate,0.000139,Legitimate
1,gojackpot,2,word,0,0.0,24,0.012481,Legitimate,0.000139,Legitimate
2,points,3,word,0,0.0,20,0.010899,Legitimate,0.000133,Legitimate
3,reminds,4,word,0,0.0,18,0.009625,Legitimate,0.000128,Legitimate
4,epicwin,5,word,0,0.0,19,0.009414,Legitimate,0.000127,Legitimate
5,login,6,word,0,0.0,12,0.009400,Legitimate,0.000127,Legitimate
6,trusted,7,word,0,0.0,19,0.008836,Legitimate,0.000125,Legitimate
7,tangkilikin,8,word,0,0.0,15,0.007871,Legitimate,0.000120,Legitimate
8,soon,9,word,0,0.0,13,0.007684,Legitimate,0.000119,Legitimate
9,smart,10,word,0,0.0,12,0.007489,Legitimate,0.000118,Legitimate


In [ ]:
i = 3                                           
print(test_data['text'][i], '| actual:', TARGET_NAMES[y_test[i]])
explain_message_nb(cnb, X_test[i], names, top_n=10)

bigwln nanalo prize moneytoken magparehistro makakuha moneytoken maglaro moneytoken makakuha moneytoken libreng moneytoken unang deposito gdubw social | actual: Legitimate


,feature,value,contribution,toward
0,makakuha moneytoken,0.382337,-0.000042,Smishing
1,magparehistro,0.211028,-0.000037,Smishing
2,magparehistro makakuha,0.252912,-0.000034,Smishing
3,moneytoken makakuha,0.266842,-0.000033,Smishing
4,moneytoken makakuha moneytoken,0.266842,-0.000033,Smishing
5,prize,0.259378,0.000027,Legitimate
6,libreng moneytoken,0.233279,-0.000025,Smishing
7,prize moneytoken,0.286475,0.000022,Legitimate
8,libreng,0.187323,-0.000015,Smishing
9,deposito,0.192252,0.000011,Legitimate


In [18]:
i = 3                                           
print(test_data['text'][i], '| actual:', TARGET_NAMES[y_test[i]])
explain_message_nb(mnb, X_test[i], names, top_n=10)

bigwln nanalo prize moneytoken magparehistro makakuha moneytoken maglaro moneytoken makakuha moneytoken libreng moneytoken unang deposito gdubw social | actual: Legitimate


,feature,value,contribution,toward
0,makakuha moneytoken,0.382337,-1.025816,Smishing
1,magparehistro,0.211028,-0.902754,Smishing
2,magparehistro makakuha,0.252912,-0.839681,Smishing
3,moneytoken makakuha,0.266842,-0.807642,Smishing
4,moneytoken makakuha moneytoken,0.266842,-0.807642,Smishing
5,libreng moneytoken,0.233279,-0.626739,Smishing
6,prize,0.259378,0.625326,Legitimate
7,prize moneytoken,0.286475,0.503166,Legitimate
8,libreng,0.187323,-0.380568,Smishing
9,deposito,0.192252,0.265935,Legitimate


## Random Forest Classifier - Fetaure Importance

In [12]:
rf_imp  = rf_importance_table(rf, feature_names=names, stats=stats, top_n=15, max_ngram=1)
rf_perm = rf_permutation_table(rf, X_test, y_test, feature_names=names,
                               stats=stats, top_n=15, max_ngram=1)
rf_perm.head(10)

,feature,rank,kind,impurity_importance,drop_macro_f1,drop_macro_f1_std,drop_f1_Smishing,drop_f1_Legitimate,docs_Smishing,mean_Smishing,docs_Legitimate,mean_Legitimate,lean,class
0,bonus,1,word,0.032322,0.064161,0.016075,0.080938,0.047384,10,0.004227,184,0.060172,Legitimate,Legitimate
1,gojackpot,2,word,0.003643,0.004919,0.006565,0.006747,0.003092,0,0.000000,24,0.012481,Legitimate,Legitimate
2,deposit,3,word,0.005442,0.004905,0.004005,0.006706,0.003104,16,0.010479,62,0.024912,Legitimate,Legitimate
3,win,4,word,0.003074,0.001635,0.003270,0.002235,0.001035,9,0.005897,32,0.016230,Legitimate,Legitimate
4,please,5,word,0.003366,0.001635,0.003270,0.002235,0.001035,14,0.008763,43,0.017597,Legitimate,Legitimate
5,gcash,6,word,0.003690,0.001635,0.003270,0.002235,0.001035,6,0.003561,29,0.017310,Legitimate,Legitimate
6,click,7,word,0.002251,0.001635,0.003270,0.002235,0.001035,12,0.010236,24,0.012170,Legitimate,Legitimate
7,de,8,word,0.001744,0.001635,0.003270,0.002235,0.001035,8,0.005474,31,0.013040,Legitimate,Legitimate
8,enjoy,9,word,0.002088,0.001635,0.003270,0.002235,0.001035,3,0.002876,24,0.012689,Legitimate,Legitimate
9,payday,10,word,0.002154,0.001635,0.003270,0.002235,0.001035,0,0.000000,15,0.007475,Legitimate,Legitimate


## Model Evaluation - Agreements across Models

In [16]:
consensus_words(nb_word_table(cnb, feature_names=names, stats=stats, top_n=30, max_ngram=1),
                rf_importance_table(rf, feature_names=names, stats=stats, top_n=30, max_ngram=1))

{'Legitimate': ['bonus', 'gojackpot', 'login', 'payday'],
 'Smishing': ['events', 'maglagay', 'magparehistro', 'tumaya', 'tutor']}

## Predictions and Confident Mistakes

In [21]:
pt = {n: prediction_table(m, X_test, y_test, test_data['text'])
      for n, m in [('MNB', mnb), ('CNB', cnb), ('RF', rf)]}

pt['RF'][pt['RF'].correct == 0].sort_values('confidence', ascending=False)

,text,actual,predicted,p_Smishing,p_Legitimate,confidence,correct
42,magparehistro libreng moneytoken tumaya pint t...,Legitimate,Smishing,0.984125,0.015875,0.984125,0
11,winnerabonus uk claim nanalo moneytoken maglag...,Legitimate,Smishing,0.856192,0.143808,0.856192,0
14,phprize uk claim nanalo moneytoken kunin inyon...,Legitimate,Smishing,0.737514,0.262486,0.737514,0
65,cassenatrundling x jsrxpckayj dagdag iyong una...,Smishing,Legitimate,0.301437,0.698563,0.698563,0
0,fex parcel p iisddcddtba reach please contact ...,Legitimate,Smishing,0.623154,0.376846,0.623154,0
39,get commission per day come give right bonusio...,Legitimate,Smishing,0.619732,0.380268,0.619732,0
3,bigwln nanalo prize moneytoken magparehistro m...,Legitimate,Smishing,0.617218,0.382782,0.617218,0
58,luckwin tv fslqau libreng pera libreng pagkaka...,Legitimate,Smishing,0.604343,0.395657,0.604343,0
90,package arrived warehouse cannot delivered due...,Smishing,Legitimate,0.397090,0.602910,0.602910,0
126,g claim moneytoken siot bonus celebrate world ...,Legitimate,Smishing,0.598877,0.401123,0.598877,0
